# Autoresearch 实验分析

对 `results.tsv` 中自主超参数调优结果的分析。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取 TSV（制表符分隔，共 5 列：commit、val_bpb、memory_gb、status、description）
df = pd.read_csv("results.tsv", sep="\t")
df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"实验总数: {len(df)}")
print(f"列名: {list(df.columns)}")
df.head(10)


In [ ]:
counts = df["status"].value_counts()
print("实验结果分布:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\n保留率: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")


In [ ]:
# 显示所有被保留（KEEP）的实验，也就是最终留下来的改进
kept = df[df["status"] == "KEEP"].copy()
print(f"保留实验（共 {len(kept)} 个）:\n")
for i, row in kept.iterrows():
    bpb = row["val_bpb"]
    desc = row["description"]
    print(f"  #{i:3d}  bpb={bpb:.6f}  mem={row['memory_gb']:.1f}GB  {desc}")


## Val BPB 随时间变化

跟踪最佳（被保留的）`val_bpb` 如何随着实验推进而演化。运行最小值代表当前的“前沿”，也就是到目前为止取得的最好结果。


In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# 为绘图过滤掉崩溃的实验
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_bpb = valid.loc[0, "val_bpb"]

# 只绘制不高于 baseline 太多的点（更有信息量的区域）
below = valid[valid["val_bpb"] <= baseline_bpb + 0.0005]

# 将被丢弃的实验画成浅色背景点
disc = below[below["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_bpb"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="丢弃")

# 将被保留的实验画成更醒目的绿色点
kept_v = below[below["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_bpb"],
           c="#2ecc71", s=50, zorder=4, label="保留", edgecolors="black", linewidths=0.5)

# 运行最小值阶梯线
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_bpb = valid.loc[kept_mask, "val_bpb"]
running_min = kept_bpb.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="当前最优")

# 给每个保留实验标注描述
for idx, bpb in zip(kept_idx, kept_bpb):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, bpb),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("实验编号", fontsize=12)
ax.set_ylabel("验证 BPB（越低越好）", fontsize=12)
ax.set_title(f"Autoresearch 进展：{n_total} 次实验，{n_kept} 次保留改进", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y 轴范围：从略低于最佳值到略高于 baseline
best_bpb = kept_bpb.min()
margin = (baseline_bpb - best_bpb) * 0.15
ax.set_ylim(best_bpb - margin, baseline_bpb + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("已保存到 progress.png")


## 汇总统计


In [ ]:
# 汇总统计
kept = df[df["status"] == "KEEP"].copy()
baseline_bpb = df.iloc[0]["val_bpb"]
best_bpb = kept["val_bpb"].min()
best_row = kept.loc[kept["val_bpb"].idxmin()]

print(f"Baseline val_bpb:  {baseline_bpb:.6f}")
print(f"最佳 val_bpb:      {best_bpb:.6f}")
print(f"总改进幅度:       {baseline_bpb - best_bpb:.6f} ({(baseline_bpb - best_bpb) / baseline_bpb * 100:.2f}%)")
print(f"最佳实验:         {best_row['description']}")
print()

# 找到每一次改进用了多少实验
print("每次改进对应的累计实验成本:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  实验 #{row['index']:3d}: bpb={row['val_bpb']:.6f}  {desc}")


## 最佳命中（按改进幅度排序的保留实验）


In [ ]:
# 每个保留实验的增量，都是相对于上一个保留实验的 bpb 计算
# （因为实验是累积推进的，每次都建立在上一个保留状态之上）
kept = df[df["status"] == "KEEP"].copy()
kept["prev_bpb"] = kept["val_bpb"].shift(1)
kept["delta"] = kept["prev_bpb"] - kept["val_bpb"]

# 去掉 baseline（没有 delta）
hits = kept.iloc[1:].copy()

# 按改进量排序（最大的排前面）
hits = hits.sort_values("delta", ascending=False)

print("排名  改进量       BPB         描述")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_bpb']:.6f}  {row['description']}")

print(f"\n总计  {hits['delta'].sum():+.6f}              相对 baseline 的总改进")
